# Fine-tuning bart-base model dengan custom tokenizer **detiknews**

In [1]:
!pip install evaluate
!pip install rouge-score
!pip install bert_score
!pip install datasets
!pip install hf_xetimport

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn

In [2]:
import pandas as pd
import numpy as np
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset, Dataset
import torch
import evaluate
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

2025-05-29 17:32:14.722950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748539934.913240      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748539934.968864      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("jawahirul/mix-datasets-8k")

# print("Path to dataset files:", path)

# Tokenize dataset

In [4]:
# Load tokenizer
tokenizer = BartTokenizer.from_pretrained("/kaggle/input/tokenizer-detiknews-pemilu/transformers/tokenizer-detiknews-pemilu-50265/1")

In [5]:
# Fungsi tokenisasi
max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
  inputs = examples['text']
  targets = examples['summary']

  model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
  labels = tokenizer(targets, max_length=max_target_length, truncation=True)

  model_inputs['labels'] = labels['input_ids']

  return model_inputs

# Compute Metrics

In [6]:
rouge_metric = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Mengganti -100 dengan pad token id untuk decoder
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # ROUGE expects a newline after each sentence
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]
    
    # ROUGE metrics
    rouge_output = rouge_metric.compute(
        predictions=decoded_preds, 
        references=decoded_labels, 
        use_stemmer=False
    )
    
    rouge_results = {
        'rouge1' : round(rouge_output['rouge1']*100, 2),
        'rouge2' : round(rouge_output['rouge2']*100, 2)
    }
    
    # BERTScore metrics
    bert_output = bertscore.compute(
        predictions=decoded_preds, 
        references=decoded_labels, 
        lang="id"  # Untuk bahasa Indonesia
    )
    bert_results = {
        "bertscore_precision": round(np.mean(bert_output["precision"]) * 100, 2),
        "bertscore_recall": round(np.mean(bert_output["recall"]) * 100, 2),
        "bertscore_f1": round(np.mean(bert_output["f1"]) * 100, 2)
    }
    
    # Menggabungkan semua metrics
    all_metrics = {**rouge_results, **bert_results}
    return all_metrics
    

# Fine-tune BART Model

In [7]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="transformers.modeling_utils")
warnings.filterwarnings("ignore", category=UserWarning, module="torch.nn.parallel")

all_metrics = []

num_folds = 5

# Loop untuk setiap lipatan
for fold in range(1, num_folds + 1):
    print(f"\n=== Memproses Fold {fold} ===")

    # 1. Load data CSV untuk fold ini
    data_files = {
        "train": f"/kaggle/input/mix-datasets-8k/train_fold{fold}.csv",
        "validation": f"/kaggle/input/mix-datasets-8k/val_fold{fold}.csv",
        "test": f"/kaggle/input/mix-datasets-8k/test_fold{fold}.csv"
    }
    dataset = load_dataset("csv", data_files=data_files)

    # 2. Tokenisasi data
    tokenized_dataset = dataset.map(preprocess_function, batched=True)

    # 3. Inisialisasi model baru untuk setiap lipatan
    model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

    # 4. Argumen Training
    training_args = Seq2SeqTrainingArguments(
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        learning_rate=1e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        predict_with_generate=True,
        report_to="none",
        fp16=True,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model
    )

    # 5. Inisialisasi seq2seqTrainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset['train'],
        eval_dataset=tokenized_dataset['validation'],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # 6. Latih model
    print(f"Melatih model Fold {fold}...")
    trainer.train()

    # simpan hasil validasi pada fold ini
    print(f"Simpan hasil validasi fold {fold}...")
    validation_results = trainer.evaluate(tokenized_dataset["validation"])
    
    # 7. Evaluasi pada test set ini
    print(f"Evaluasi model test set Fold {fold}...")
    test_results = trainer.evaluate(tokenized_dataset["test"])
    print(f"Hasil Test Fold {fold}:")
    print(f"  ROUGE-1: {test_results['eval_rouge1']:.2f}")
    print(f"  ROUGE-2: {test_results['eval_rouge2']:.2f}")
    print(f"  BERTScore F1: {test_results['eval_bertscore_f1']:.2f}")

    # save metrik
    all_metrics.append({
        "fold": fold,
        "val_rouge1": validation_results["eval_rouge1"],
        "val_rouge2": validation_results["eval_rouge2"],
        "val_bertscore_precision": validation_results["eval_bertscore_precision"],
        "val_bertscore_recall": validation_results["eval_bertscore_recall"],
        "val_bertscore_f1": validation_results["eval_bertscore_f1"],
        "test_rouge1": test_results["eval_rouge1"],
        "test_rouge2": test_results["eval_rouge2"],
        "test_bertscore_precision": test_results["eval_bertscore_precision"],
        "test_bertscore_recall": test_results["eval_bertscore_recall"],
        "test_bertscore_f1": test_results["eval_bertscore_f1"]
    })

    # Simpan model setelah pelatihan
    model_save_path = f"/kaggle/working/model_fold_{fold}/"
    
    trainer.save_model(model_save_path)

    torch.cuda.empty_cache()


=== Memproses Fold 1 ===


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6400 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Melatih model Fold 5...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Bertscore Precision,Bertscore Recall,Bertscore F1
1,5.862000,5.036901,25.130000,12.580000,73.090000,69.460000,71.190000
2,5.141900,4.780811,25.220000,12.680000,73.130000,69.500000,71.240000
3,4.878000,4.675848,25.300000,12.690000,73.130000,69.470000,71.220000
4,4.729500,4.543715,25.310000,12.680000,73.220000,69.490000,71.270000
5,4.615300,4.501136,25.520000,12.860000,73.230000,69.520000,71.290000
6,4.535500,4.501770,25.780000,13.030000,73.250000,69.550000,71.320000
7,4.471900,4.423691,25.810000,12.910000,73.270000,69.570000,71.340000
8,4.430200,4.409060,25.590000,12.820000,73.220000,69.490000,71.270000
9,4.404700,4.413807,25.480000,12.780000,73.160000,69.480000,71.240000
10,4.380000,4.403267,25.610000,12.810000,73.210000,69.500000,71.270000


Simpan hasil validasi fold 5...


Evaluasi model test set Fold 5...
Hasil Test Fold 5:
  ROUGE-1: 27.06
  ROUGE-2: 14.63
  BERTScore F1: 71.91


In [8]:
# Save output metric
df = pd.DataFrame(all_metrics)

df.to_excel('detiknews-all_metric.xlsx', index=False)